In [1]:
from pathlib import Path

def find_project_root(start: Path, markers=("pyproject.toml", ".git")):
    for p in [start] + list(start.parents):
        if any((p / m).exists() for m in markers):
            return p
    raise RuntimeError("Project root not found")

ROOT = find_project_root(Path.cwd())

We start by substituting certain Doric forms for their Attic-Alexandrian  equivalents

In [2]:
from lxml import etree
import os
from pathlib import Path
import re
from tqdm import tqdm

from grc_utils import normalize_word

def replace_word_in_syllables(search_word: str, replace_word: str, xml_path: Path, output_path: Path) -> bool:
    """
    Replace a word in XML while preserving syllable structure.
    Words are matched with word boundaries (spaces).
    Saves to the specified output_path (required for safety).
    Returns True if replacements were made, False otherwise.
    """
    if output_path is None:
        raise ValueError("output_path is required (specify where to save the modified file)")
    
    with open(xml_path, 'rb') as f:
        tree = etree.parse(f)
    
    root = tree.getroot()
    replacements_made = False
    
    # Normalize search term for matching
    search_word_normalized = normalize_word(search_word)
    
    for l_elem in root.iter('l'):
        # Collect all <syll> elements and their text
        syll_elements = list(l_elem.findall('syll'))
        if not syll_elements:
            continue
        
        # Reconstruct full text with position tracking
        full_text = ''.join(syll.text or '' for syll in syll_elements)
        full_text_normalized = normalize_word(full_text)
        
        # Only replace complete words (with word boundaries)
        pattern = r'\b' + re.escape(search_word_normalized) + r'\b'
        new_text_normalized = re.sub(pattern, normalize_word(replace_word), full_text_normalized)
        
        if new_text_normalized != full_text_normalized:
            replacements_made = True
            # Redistribute the new text across syllables
            char_idx = 0
            for syll in syll_elements:
                old_text = syll.text or ''
                syll_len = len(old_text)
                syll.text = new_text_normalized[char_idx:char_idx + syll_len]
                char_idx += syll_len
    
    if replacements_made:
        with open(output_path, 'wb') as f:
            tree.write(f, encoding='utf-8', xml_declaration=True, pretty_print=True)
    
    return replacements_made

# Process all XML files in the triads directory
triad_path = ROOT / "data/compiled/triads/"
output_doric_triad_path = ROOT / "data/compiled_doric/triads/"

# Ensure output directory exists
output_doric_triad_path.mkdir(parents=True, exist_ok=True)

# Dictionary of Doric/Attic-Alexandrian word substitutions
replacements = {
    "Αἰγιμιοῦ": "Αἰγιμίου",
    "Ἀλφεοῦ": "Ἀλφέου",
    "Δαναοῦ": "Δανάου",
}

triad_files = sorted([triad_path / f for f in os.listdir(triad_path) if f.endswith('.xml')])

print(f"Processing {len(triad_files)} files...")
print(f"Substitutions to perform: {list(replacements.items())}\n")

# Track substitutions per word
substitution_counts = {search: 0 for search in replacements}
total_files_modified = 0

for xml_file in tqdm(triad_files):
    output_file = output_doric_triad_path / xml_file.name
    file_modified = False
    
    for search_word, replace_word in replacements.items():
        if replace_word_in_syllables(search_word, replace_word, xml_file, output_file):
            substitution_counts[search_word] += 1
            file_modified = True
    
    if file_modified:
        total_files_modified += 1
        print(f"✓ {xml_file.name}: modified")

print(f"\n{'='*60}")
print(f"Summary: {total_files_modified} file(s) modified")
print(f"{'='*60}")
for search_word, count in substitution_counts.items():
    replace_word = replacements[search_word]
    print(f"'{search_word}' → '{replace_word}': {count} substitution(s)")
print(f"{'='*60}")
print(f"Output saved to: {output_doric_triad_path}")


Processing 4 files...
Substitutions to perform: [('Αἰγιμιοῦ', 'Αἰγιμίου'), ('Ἀλφεοῦ', 'Ἀλφέου'), ('Δαναοῦ', 'Δανάου')]



 50%|█████     | 2/4 [00:00<00:00, 19.58it/s]

✓ ht_isthmians_triads.xml: modified
✓ ht_nemeans_triads.xml: modified
✓ ht_olympians_triads.xml: modified
✓ ht_pythians_triads.xml: modified

100%|██████████| 4/4 [00:00<00:00, 16.15it/s]



Summary: 4 file(s) modified
'Αἰγιμιοῦ' → 'Αἰγιμίου': 1 substitution(s)
'Ἀλφεοῦ' → 'Ἀλφέου': 3 substitution(s)
'Δαναοῦ' → 'Δανάου': 1 substitution(s)
Output saved to: /Users/albin/git/gh/responsio-accentuum/data/compiled_doric/triads


In [4]:
from statistics import mean

from responsio_accentuum import compatibility_corpus, compatibility_ratios_to_stats

output_doric_triad_path = ROOT / "data/compiled_doric/triads/"

# Position-level observed statistic

scores_doric = compatibility_corpus(output_doric_triad_path)
T_obs_pos_triads_doric = compatibility_ratios_to_stats(scores_doric)

# Song-level observed statistic

for collection in scores_doric:
    odes_averages = []
    for ode in collection:
        odes_averages.append(compatibility_ratios_to_stats(ode))

T_obs_song_triads_doric = mean(odes_averages)

print(f"\nDoric Triads T_pos: {T_obs_pos_triads_doric:.4f}")
print(f"Doric Triads T_song: {T_obs_song_triads_doric:.4f}")


 25%|██▌       | 1/4 [00:00<?, ?it/s]

5it [00:00,  4.78it/s]                       


Doric Triads T_pos: 0.5307
Doric Triads T_song: 0.5433
